# Model Predictions (coupler_NCap_cap_matrix)

## Configuration

In [1]:
# The parameter file is where the hyperparameters are set. 
# It's reccomended to look at that file first, its interesting and you can set stuff there

from parameters import *

## Library

In [2]:
# Disable some console warnings
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf# Disable some console warnings so you can be free of them printing. 
# Comment the next two lines if you are a professional and like looking at warnings.
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

## Dataset

### Load

In [3]:
def load_scaled_split(kind, split):
    # kind: 'one_hot' or 'linear'
    return np.load(f"{DATA_DIR}/npy/{split}_{kind}_encoding_scaled.npy", allow_pickle=True)

if 'Try Both' in ENCODING_TYPE:
    # one-hot
    X_train_one_hot_encoding = load_scaled_split('one_hot', 'x_train')
    X_val_one_hot_encoding   = load_scaled_split('one_hot', 'x_val')
    X_test_one_hot_encoding  = load_scaled_split('one_hot', 'x_test')

    y_train_one_hot_encoding = load_scaled_split('one_hot', 'y_train')
    y_val_one_hot_encoding   = load_scaled_split('one_hot', 'y_val')
    y_test_one_hot_encoding  = load_scaled_split('one_hot', 'y_test')

    # linear
    X_train_linear_encoding = load_scaled_split('linear', 'x_train')
    X_val_linear_encoding   = load_scaled_split('linear', 'x_val')
    X_test_linear_encoding  = load_scaled_split('linear', 'x_test')

    y_train_linear_encoding = load_scaled_split('linear', 'y_train')
    y_val_linear_encoding   = load_scaled_split('linear', 'y_val')
    y_test_linear_encoding  = load_scaled_split('linear', 'y_test')

else:
    if 'one hot' in ENCODING_TYPE:
        kind = 'one_hot'
    elif 'Linear' in ENCODING_TYPE:
        kind = 'linear'
    else:
        raise ValueError(f"Unknown ENCODING_TYPE: {ENCODING_TYPE}")

    X_train = load_scaled_split(kind, 'x_train')
    X_val   = load_scaled_split(kind, 'x_val')
    X_test  = load_scaled_split(kind, 'x_test')

    y_train = load_scaled_split(kind, 'y_train')
    y_val   = load_scaled_split(kind, 'y_val')
    y_test  = load_scaled_split(kind, 'y_test')


### Visualize

In [4]:
# Decide which model file & test set to use
chosen_path = "model/mlp_6_264_464_364_364_864_13_best_model.keras"      
X_test_cur = np.asarray(X_test)
y_test_cur = np.asarray(y_test)
encoding = ENCODING_TYPE.replace(' ','_')
y_encoding_format_name = encoding    

# Load y headers for labeling columns
y_headers_csv = f'y_characteristics_{y_encoding_format_name}_encoding.csv'
with open(y_headers_csv, 'r') as f:
    headers = f.readline().strip().split(',')

In [5]:
#run on CPU
tf.keras.backend.clear_session()
gc.collect()
try:
    tf.config.experimental.reset_memory_stats('GPU:0')
except Exception:
    pass

with tf.device('/CPU:0'):
    chosen_model = load_model(chosen_path, compile=False)  #dont compile it because we just need to predict
    y_pred = chosen_model.predict(X_test_cur, verbose=0)

print(f"\n—— {os.path.basename(chosen_path)} ——")
chosen_model.summary()
print(f"Samples: {len(X_test_cur)} | Targets dim: {y_test_cur.shape[1]}")

I0000 00:00:1770826220.150066 3344567 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1056 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe MIG 4g.40gb, pci bus id: 0000:00:11.0, compute capability: 8.0



—— mlp_6_264_464_364_364_864_13_best_model.keras ——


I0000 00:00:1770826220.518894 3448731 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc0 (Dense)                     │ (None, 264)            │         1,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu0 (LeakyReLU)         │ (None, 264)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout0 (Dropout)              │ (None, 264)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 464)            │       122,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu1 (LeakyReLU)         │ (None, 464)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 464)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 364)            │       169,260 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu2 (LeakyReLU)         │ (None, 364)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 364)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc3 (Dense)                     │ (None, 364)            │       132,860 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu3 (LeakyReLU)         │ (None, 364)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout3 (Dropout)              │ (None, 364)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc4 (Dense)                     │ (None, 864)            │       315,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu4 (LeakyReLU)         │ (None, 864)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout4 (Dropout)              │ (None, 864)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_output (Dense)               │ (None, 13)             │        11,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 753,533 (2.87 MB)

 Trainable params: 753,533 (2.87 MB)

 Non-trainable params: 0 (0.00 B)

Samples: 65 | Targets dim: 13


# Scaled

In [6]:
#use a smaller view if you want
N_SAMPLES_TO_SHOW = 50


n_samples = min(N_SAMPLES_TO_SHOW, len(X_test_cur))
n_params  = y_test_cur.shape[1]

# scaled errors
sq_errors  = (y_test_cur - y_pred) ** 2
abs_errors = np.abs(y_test_cur - y_pred)

# scaled dataframe
rows = []
for i in range(n_samples):
    top_to_top, top_to_bottom, top_to_ground, bottom_to_bottom, bottom_to_ground, ground_to_ground = X_test_cur[i, 0], X_test_cur[i, 1], X_test_cur[i, 2],X_test_cur[i, 3],X_test_cur[i, 4],X_test_cur[i, 5]
    for j in range(n_params):
        rows.append({
            "sample_idx": i,
            "top_to_bottom": top_to_bottom,
            "top_to_ground": top_to_ground,
            "bottom_to_bottom": bottom_to_bottom,
            "bottom_to_ground": bottom_to_ground,
            "ground_to_ground": ground_to_ground,
            "param": headers[j],
            "ref":  y_test_cur[i, j],
            "pred": y_pred[i, j],
            "abs_error": abs_errors[i, j],
            "sq_error":  sq_errors[i, j],
        })
df = pd.DataFrame(rows)

# save scaled predictions
out_csv = Path(f"predictions_and_errors_{y_encoding_format_name}.csv")
df.to_csv(out_csv, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv.resolve()}\n")

#pretty print per-sample (scaled)
for i in range(n_samples):
    sub = df[df["sample_idx"] == i].copy()
    sub = sub[["param", "ref", "pred", "abs_error", "sq_error"]]
    header_line = (
        f"— Sample {i} — "
        f"X: top_to_top={X_test_cur[i,0]:.9g}, top_to_bottom={X_test_cur[i,1]:.9g}, top_to_ground={X_test_cur[i,2]:.9g}, bottom_to_bottom={X_test_cur[i,3]:.9g},bottom_to_ground={X_test_cur[i,4]:.9g},ground_to_ground={X_test_cur[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

#print global stats (scaled)
print("Global scaled error stats:")
print("  min abs_error:", float(abs_errors.min()))
print("  median abs_error:", float(np.median(abs_errors)))
print("  max abs_error:", float(abs_errors.max()))
print("\nHere onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably throwing off the global average. These will be rounded in the future and will probably always  round to the right number to reconstruct the correct category-- but for now it might throw off  the overall average error. In the future we might want to just have it consider the non-categorical data when finding an overall average and reporting that number.\n")


Saved CSV -> /home/olivias/ML_qubit_design/model_predict_coupler_NCap_cap_matrix/predictions_and_errors_one_hot.csv

— Sample 0 — X: top_to_top=0.00832192455, top_to_bottom=0.00565011737, top_to_ground=0.0365676091, bottom_to_bottom=0.00692052511,bottom_to_ground=0.00850165266,ground_to_ground=0.0110262114
                         param  ref      pred  abs_error     sq_error
        design_options.cap_gap  0.0  0.002884   0.002884 8.316943e-06
      design_options.cap_width  0.2  0.052185   0.147815 2.184926e-02
  design_options.finger_length  0.0  0.048798   0.048798 2.381206e-03
 design_options.finger_count_1  1.0  0.925293   0.074707 5.581141e-03
 design_options.finger_count_2  0.0  0.001198   0.001198 1.434761e-06
 design_options.finger_count_3  0.0 -0.000698   0.000698 4.873291e-07
 design_options.finger_count_4  0.0  0.001030   0.001030 1.060835e-06
 design_options.finger_count_5  0.0 -0.001871   0.001871 3.501049e-06
 design_options.finger_count_6  0.0  0.000640   0.000640 4.10

# Unscaled

In [7]:
#unscale everything and look at errors again. 
#You can compare the unscaled actual values to the ml_00...py notebook to convice yourself that unscaling worked

with open('X_names', 'r') as f:
    X_index_names = f.read().splitlines()

# Load headers from the saved numpy file (same ordering as y arrays)
headers = np.load("y_columns.npy", allow_pickle=True).astype(str).tolist()

# --- detect one-hot columns for finger_count and prep decoding ---
fc_base = "design_options.finger_count"
fc_prefix = fc_base + "_"

fc_idx = [j for j, h in enumerate(headers) if h.startswith(fc_prefix)]
fc_class_values = []
for j in fc_idx:
    try:
        fc_class_values.append(int(headers[j].split(fc_prefix, 1)[1]))
    except Exception:
        fc_class_values.append(None)

# unscaling x
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        scaler = joblib.load(f'scalers/scaler_X_{X_index_names[j]}.save')
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]

# unscaling y
y_test_unscaled = np.asarray(y_test_cur.copy())
for i in range(y_test_unscaled.shape[0]):
    for j in range(y_test_unscaled.shape[1]):
        # one-hot columns don't need unscaling (and often no scaler exists / it is identity anyway)
        if j in fc_idx:
            continue
        scaler = joblib.load(f'scalers/scaler_y_{headers[j]}_{y_encoding_format_name}_encoding.save')
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]

# unscaling y predictions
y_pred_unscaled = np.asarray(y_pred.copy())
for i in range(y_pred_unscaled.shape[0]):
    for j in range(y_pred_unscaled.shape[1]):
        if j in fc_idx:
            continue
        scaler = joblib.load(f'scalers/scaler_y_{headers[j]}_{y_encoding_format_name}_encoding.save')
        y_pred_unscaled[i, j] = scaler.inverse_transform([[y_pred_unscaled[i, j]]])[0][0]

n_samples, n_params = y_test_unscaled.shape
 

# find how good or bad we did (the errors)
sq_errors_unscaled  = (y_test_unscaled - y_pred_unscaled) ** 2
abs_errors_unscaled = np.abs(y_test_unscaled - y_pred_unscaled)

# making a nice fancy dataframe, we like fancy things
rows_unscaled = []
for i in range(N_SAMPLES_TO_SHOW):
    qubit_frequency_GHz, anharmonicity_MHz = X_test_unscaled[i, 0], X_test_unscaled[i, 1]

    # continuous + non-finger_count one-hot outputs
    for j in range(n_params):
        if j in fc_idx:
            continue
        rows_unscaled.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": headers[j],
            "ref_unscaled": y_test_unscaled[i, j],
            "pred_unscaled": y_pred_unscaled[i, j],
            "abs_error_unscaled": abs_errors_unscaled[i, j],
            "sq_error_unscaled": sq_errors_unscaled[i, j],
        })

    # collapsed finger_count (decode by argmax)
    if len(fc_idx) > 0 and all(v is not None for v in fc_class_values):
        ref_vec = y_test_unscaled[i, fc_idx]
        pred_vec = y_pred_unscaled[i, fc_idx]

        ref_k = int(np.argmax(ref_vec))
        pred_k = int(np.argmax(pred_vec))

        ref_fc = int(fc_class_values[ref_k])
        pred_fc = int(fc_class_values[pred_k])

        abs_fc = float(abs(ref_fc - pred_fc))
        sq_fc = float((ref_fc - pred_fc) ** 2)

        rows_unscaled.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": fc_base,
            "ref_unscaled": ref_fc,
            "pred_unscaled": pred_fc,
            "abs_error_unscaled": abs_fc,
            "sq_error_unscaled": sq_fc,
        })

df_unscaled = pd.DataFrame(rows_unscaled)

# save csv of unscaled results uncase we lose this notebook due to github blowing up, ya never know
out_csv_unscaled = Path(f"predictions_and_errors_unscaled_{y_encoding_format_name}.csv")
df_unscaled.to_csv(out_csv_unscaled, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv_unscaled.resolve()}\n")

# print out stuff so you can see it here if you are to lazy like me to open a csv
for i in range(N_SAMPLES_TO_SHOW):
    sub = df_unscaled[df_unscaled["sample_idx"] == i].copy()
    sub = sub[["param", "ref_unscaled", "pred_unscaled", "abs_error_unscaled", "sq_error_unscaled"]]
    header_line = (
        f"— Sample {i} (Unscaled) — "
        f"X: cross_to_ground={X_test_unscaled[i,0]:.9g}, claw_to_ground={X_test_unscaled[i,1]:.9g}, cross_to_claw={X_test_unscaled[i,2]:.9g}, cross_to_cross={X_test_unscaled[i,3]:.9g}, claw_to_claw={X_test_unscaled[i,4]:.9g}, ground_to_ground={X_test_unscaled[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

# look at overall stats, see below comment for a caviat 
# (exclude finger_count one-hot columns from global stats so they don't skew the averages)
if len(fc_idx) > 0:
    mask = np.ones(n_params, dtype=bool)
    mask[fc_idx] = False
    abs_for_stats = abs_errors_unscaled[:, mask]
else:
    abs_for_stats = abs_errors_unscaled

print("Global unscaled error stats:")
print("  min abs_error:", float(abs_for_stats.min()))
print("  median abs_error:", float(np.median(abs_for_stats)))
print("  max abs_error:", float(abs_for_stats.max()))

'''
Here onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably 
throwing off the global average. These will be rounded in the future and will probably always 
round to the right number to reconstruct the correct category-- but for now it might throw off 
the overall average error. In the future we might want to just have it consider the non-categorical 
data when finding an overall average and reporting that number.
'''



Saved CSV -> /home/olivias/ML_qubit_design/model_predict_coupler_NCap_cap_matrix/predictions_and_errors_unscaled_one_hot.csv

— Sample 0 (Unscaled) — X: cross_to_ground=14.8635827, claw_to_ground=0.668175255, cross_to_claw=13.854644, cross_to_cross=13.62255, claw_to_claw=12.7645607, ground_to_ground=58.0137
                       param  ref_unscaled  pred_unscaled  abs_error_unscaled  sq_error_unscaled
      design_options.cap_gap      0.000002       0.000002        1.383743e-08       1.914745e-16
    design_options.cap_width      0.000007       0.000005        1.475977e-06       2.178509e-12
design_options.finger_length      0.000016       0.000017        1.444952e-06       2.087885e-12
 design_options.finger_count      1.000000       1.000000        0.000000e+00       0.000000e+00

— Sample 1 (Unscaled) — X: cross_to_ground=21.5988461, claw_to_ground=3.83709971, cross_to_claw=17.3630789, cross_to_cross=27.51404, claw_to_claw=23.2719519, ground_to_ground=73.35986
                    

'\nHere onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably \nthrowing off the global average. These will be rounded in the future and will probably always \nround to the right number to reconstruct the correct category-- but for now it might throw off \nthe overall average error. In the future we might want to just have it consider the non-categorical \ndata when finding an overall average and reporting that number.\n'

In [8]:
m = tf.keras.models.load_model(chosen_path, compile=True)  # loads optimizer too (if saved)

# ---- neurons per layer (your original logic, slightly generalized) ----
neurons_per_layer = [
    l.units for l in m.layers
    if isinstance(l, tf.keras.layers.Dense) and l.name.startswith("fc")
]
print("Best NEURONS_PER_LAYER:", neurons_per_layer)

# ---- dropout rate(s) from Dropout layers ----
dropouts = [
    (l.name, float(l.rate)) for l in m.layers
    if isinstance(l, tf.keras.layers.Dropout)
]
if dropouts:
    print("Best DROPOUT rate(s):", dropouts)
else:
    print("Best DROPOUT rate(s): none found")

# ---- learning rate / schedule details ----
opt = getattr(m, "optimizer", None)
if opt is None:
    print("Best learning rate: (no optimizer found on loaded model)")
else:
    lr_obj = opt.learning_rate  # can be float/Variable OR a schedule

    # Case 1: schedule (e.g., ExponentialDecay)
    if isinstance(lr_obj, tf.keras.optimizers.schedules.LearningRateSchedule):
        print("Best learning rate: (schedule)", type(lr_obj).__name__)

        # Try to print schedule config nicely
        try:
            cfg = lr_obj.get_config()
            print("LR schedule config:")
            for k, v in sorted(cfg.items()):
                print(f"  {k}: {v}")
        except Exception as e:
            print("Could not read LR schedule config:", repr(e))

        # Also print the current LR value at step=optimizer.iterations
        try:
            step = int(tf.keras.backend.get_value(opt.iterations))
            current_lr = float(lr_obj(step).numpy())
            print(f"Current LR at step {step}:", current_lr)
        except Exception as e:
            print("Could not compute current LR from schedule:", repr(e))

    # Case 2: plain numeric / variable LR
    else:
        try:
            lr_val = float(tf.keras.backend.get_value(lr_obj))
            print("Best learning rate:", lr_val)
        except Exception as e:
            print("Best learning rate: (unreadable)", repr(e))


/home/olivias/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 25 variables whereas the saved optimizer has 29 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Best NEURONS_PER_LAYER: [264, 464, 364, 364, 864, 13]
Best DROPOUT rate(s): [('dropout0', 0.06), ('dropout1', 0.06), ('dropout2', 0.06), ('dropout3', 0.06), ('dropout4', 0.06)]
Best learning rate: 0.0009287500288337469
